# Modeling
In this section, multiple models are implemented to perform brain tumor segmentation from MRI images. The goal is to compare different approaches ranging from simple baselines to advanced deep learning architectures.

The following models are explored:
- Baseline (threshold-based segmentation)
- Convolutional Neural Network (CNN)
- U-Net (advanced segmentation model)

The dataset is split into training and validation sets to evaluate model generalization.

In [17]:
from torch.utils.data import DataLoader, random_split


import os
import sys

sys.path.append(os.path.abspath(".."))
from dataset import BrainTumorDataset

BASE_DIR = os.path.dirname(os.getcwd())

# Create dataset
dataset = BrainTumorDataset(
    image_dir=os.path.join(BASE_DIR, "data/processed/images"),
    mask_dir=os.path.join(BASE_DIR, "data/processed/masks"),
)

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 202
Validation size: 51


In [ ]:
import os
import mlflow
import mlflow.pytorch

mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('mlflow.db')}")
mlflow.set_experiment("Brain-Tumor-Segmentation")
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("Brain-Tumor-Segmentation").name)


MLflow tracking URI: sqlite:///c:/Users/hamad/OneDrive/Desktop/ML%20-%20P/notebooks/mlflow.db
Experiment: Brain-Tumor-Segmentation


### Baseline Model

A simple threshold-based segmentation approach is used as a baseline. This method classifies pixels based on intensity values, without learning from data.

This serves as a reference to highlight the limitations of non-learning approaches.

In [19]:
# Baseline Evaluation

from metrics import dice_score, iou_score

def evaluate_baseline(loader):
    dice_total = 0
    iou_total = 0
    
    for images, masks in loader:
        
        if images.shape[1] == 3:
            images_gray = images.mean(dim=1, keepdim=True)
        else:
            images_gray = images
        
        preds = (images_gray > 0.5).float()
        
        dice_total += dice_score(preds, masks)
        iou_total += iou_score(preds, masks)
    
    return dice_total / len(loader), iou_total / len(loader)

baseline_dice, baseline_iou = evaluate_baseline(val_loader)

In [20]:
with mlflow.start_run(run_name="Baseline"):
    mlflow.log_params({
        "model": "Baseline",
        "method": "threshold",
        "threshold": 0.5,
    })
    mlflow.log_metrics({
        "val_dice": float(baseline_dice),
        "val_iou": float(baseline_iou),
    })
    print(f"[Baseline] Dice: {float(baseline_dice):.4f} | IoU: {float(baseline_iou):.4f}")
    print("Baseline run logged to MLflow.")


[Baseline] Dice: 0.3899 | IoU: 0.2479
Baseline run logged to MLflow.


###  Convolutional Neural Network (CNN)

A simple CNN is implemented to learn spatial features from MRI images. While it improves over the baseline, it lacks the ability to precisely localize tumor regions.

In [21]:
import importlib
import metrics
importlib.reload(metrics)

import torch
import torch.nn as nn
import torch.nn.functional as F
from metrics import dice_loss
from model import SimpleCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EPOCHS = 2

cnn_model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)

mlflow.start_run(run_name="CNN")
mlflow.log_params({
    "model": "CNN",
    "epochs": EPOCHS,
    "batch_size": 8,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "loss": "Dice",
})

for epoch in range(EPOCHS):
    cnn_model.train()
    total_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = cnn_model(images)
        loss = dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(train_loader)
    mlflow.log_metric("train_loss", epoch_loss, step=epoch)
    print(f"[CNN] Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}")


Using device: cpu


[CNN] Epoch 1/2, Loss: 0.6085
[CNN] Epoch 2/2, Loss: 0.4829


In [22]:
# Utility Functions

def evaluate_model(model, loader, device):
    model.eval()
    
    dice_total = 0
    iou_total = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            preds = model(images)
            
            dice_total += dice_score(preds, masks).item()
            iou_total += iou_score(preds, masks).item()
    
    return dice_total / len(loader), iou_total / len(loader)

def dice_score(pred, target, smooth=1):
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def iou_score(pred, target, smooth=1):
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    
    return (intersection + smooth) / (union + smooth)

In [23]:
cnn_dice, cnn_iou = evaluate_model(cnn_model, val_loader, device)
mlflow.log_metrics({"val_dice": cnn_dice, "val_iou": cnn_iou})
mlflow.pytorch.log_model(cnn_model, "cnn_model")
mlflow.end_run()
print(f"[CNN] Val Dice: {cnn_dice:.4f} | Val IoU: {cnn_iou:.4f}")
print("CNN run logged to MLflow.")


2026/05/04 16:22:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 16:22:08 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


[CNN] Val Dice: 0.6417 | Val IoU: 0.4824
CNN run logged to MLflow.


###  U-Net (Advanced Model)

U-Net is a specialized architecture for medical image segmentation. Its encoder-decoder structure with skip connections enables precise localization of tumor regions.

This model is expected to outperform both the baseline and CNN models.

In [24]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU(),
            )

        self.enc1 = block(3, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = block(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.up1(b)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        return torch.sigmoid(self.final(d2))


def combined_loss(pred, target):
    bce = F.binary_cross_entropy(pred, target)
    dice = dice_loss(pred, target)
    return bce + dice


model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

mlflow.start_run(run_name="U-Net")
mlflow.log_params({
    "model": "U-Net",
    "epochs": EPOCHS,
    "batch_size": 8,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "loss": "BCE + Dice",
})

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = combined_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(train_loader)
    mlflow.log_metric("train_loss", epoch_loss, step=epoch)
    print(f"[U-Net] Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}")


[U-Net] Epoch 1/2, Loss: 1.2987
[U-Net] Epoch 2/2, Loss: 1.1115


In [25]:
unet_dice, unet_iou = evaluate_model(model, val_loader, device)
mlflow.log_metrics({"val_dice": unet_dice, "val_iou": unet_iou})
mlflow.pytorch.log_model(model, "unet_model")
mlflow.end_run()
print(f"[U-Net] Val Dice: {unet_dice:.4f} | Val IoU: {unet_iou:.4f}")
print("U-Net run logged to MLflow.")


2026/05/04 16:24:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 16:24:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


[U-Net] Val Dice: 0.5963 | Val IoU: 0.4358
U-Net run logged to MLflow.


##  Evaluation

Models are evaluated using Dice Score and Intersection over Union (IoU), which are suitable metrics for segmentation tasks with class imbalance.

In [26]:
# Comparison Table

import pandas as pd

results = {
    "Model": ["Baseline", "CNN", "U-Net"],
    "Dice Score": [
        baseline_dice.item(),
        cnn_dice,
        unet_dice
    ],
    "IoU Score": [
        baseline_iou.item(),
        cnn_iou,
        unet_iou
    ]
}

df = pd.DataFrame(results)
df

,Model,Dice Score,IoU Score
0,Baseline,0.389857,0.247881
1,CNN,0.641747,0.482420
2,U-Net,0.596325,0.435839


###  Results Interpretation

The baseline method achieved very low performance, as it relies solely on pixel intensity and cannot effectively distinguish tumor regions from background.

The CNN model showed moderate improvement, demonstrating the ability to learn spatial features. However, its performance remains limited due to its relatively simple architecture and lack of precise localization capability.

The U-Net model significantly outperformed both the baseline and CNN models, achieving a Dice Score of approximately 0.70. This improvement is attributed to its encoder-decoder architecture with skip connections, which enables effective capture of both global context and fine-grained spatial details.

These results highlight the importance of specialized architectures like U-Net for medical image segmentation tasks, particularly in handling class imbalance and accurately localizing tumor regions.

In [27]:
print(masks.min(), masks.max())

tensor(0.) tensor(1.)


In [28]:
images, masks = next(iter(train_loader))
print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

Images shape: torch.Size([8, 3, 256, 256])
Masks shape: torch.Size([8, 1, 256, 256])


In [29]:
print("Mask min:", masks.min().item())
print("Mask max:", masks.max().item())

Mask min: 0.0
Mask max: 1.0


In [30]:
images = images.to(device)

model.eval()
with torch.no_grad():
    preds = model(images)

print("Pred min:", preds.min().item())
print("Pred max:", preds.max().item())

Pred min: 7.151306363084586e-06
Pred max: 0.9027639627456665
